# Imports and Functions

In [1]:
from collections import defaultdict
import json
import os
import glob
from typing import List, Callable, Union

import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, mean_absolute_error, root_mean_squared_error,
                             precision_recall_curve, r2_score, roc_auc_score, mean_absolute_percentage_error, auc)
%load_ext autoreload
%autoreload 2

from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from chemprop import models

In [2]:
def parse_indices(idxs):
    """Parses a string of indices into a list of integers. e.g. '0,1,2-4' -> [0, 1, 2, 3, 4]"""
    if isinstance(idxs, str):
        indices = []
        for idx in idxs.split(","):
            if "-" in idx:
                start, end = map(int, idx.split("-"))
                indices.extend(range(start, end + 1))
            else:
                indices.append(int(idx))
        return indices
    return idxs

def prc_auc(targets: List[int], preds: List[float]) -> float:
    """
    Computes the area under the precision-recall curve.

    :param targets: A list of binary targets.
    :param preds: A list of prediction probabilities.
    :return: The computed prc-auc.
    """
    precision, recall, _ = precision_recall_curve(targets, preds)
    return auc(recall, precision)

def get_metric_func(metric: str):
    r"""
    Gets the metric function corresponding to a given metric name.

    Supports:

    * :code:`roc-auc`: Area under the receiver operating characteristic curve
    * :code:`prc-auc`: Area under the precision recall curve
    * :code:`ap`: Average precision from prediction scores
    * :code:`rmse`: Root mean squared error
    * :code:`mae`: Mean absolute error
    * :code:`r2`: Coefficient of determination R\ :superscript:`2`

    :param metric: Metric name.
    :return: A metric function which takes as arguments a list of targets and a list of predictions and returns.
    """
    if metric == 'roc-auc':
        return roc_auc_score

    if metric == 'prc-auc':
        return prc_auc
    
    if metric == 'ap':
        return average_precision_score

    if metric == 'rmse':
        return root_mean_squared_error
    
    if metric == 'mae':
        return mean_absolute_error

    if metric == 'r2':
        return r2_score
    
    raise ValueError(f'Metric "{metric}" not supported.')
    

In [14]:
def evaluate_results_simple(data_path, splits_path, result_dir, num_tasks, metrics, target_columns=None, num_folds=5, decimal_places=4):
    """
    Simple evaluation function that computes mean ± std across folds without statistical tests.
    
    Args:
        data_path: Path to the dataset CSV file
        splits_path: Path to the multiple_splits.json file
        result_dir: Directory containing fold_X/model_Y/test_predictions.csv structure
        num_tasks: Number of target tasks
        metrics: List of metrics to compute
        target_columns: List of target column names (optional)
        num_folds: Number of folds (default: 5)
        decimal_places: Number of decimal places for formatting (default: 4)
    
    Returns:
        dict: Results with formatted mean ± std strings
    """
    df = pd.read_csv(data_path)
    with open(splits_path, "rb") as json_file:
        split_idxss = json.load(json_file)
    
    # Use the first fold's test set (all folds should have the same test set)
    test_indices = parse_indices(split_idxss[0]["test"])
    test_df = df.iloc[test_indices]
    target_columns = test_df.keys()[-num_tasks:].tolist() if target_columns is None else target_columns
    
    metric_to_func = {metric: get_metric_func(metric) for metric in metrics}
    
    # Store results for each fold
    fold_results = defaultdict(list)
    
    for fold_idx in range(num_folds):
        fold_dir = os.path.join(result_dir, f"fold_{fold_idx}")
        
        if not os.path.exists(fold_dir):
            print(f"Warning: Fold directory {fold_dir} does not exist")
            continue
            
        # Find all model directories in this fold
        model_dirs = [d for d in os.listdir(fold_dir) 
                     if os.path.isdir(os.path.join(fold_dir, d)) and d.startswith('model_')]
        
        if not model_dirs:
            print(f"Warning: No model directories found in fold {fold_idx}")
            continue
        
        # Sort model directories by model index
        model_dirs.sort(key=lambda x: int(x.split('_')[1]))
        
        # Collect predictions from all models in this fold
        df_pred_list = []
        for model_dir in model_dirs:
            model_file = os.path.join(fold_dir, model_dir, "test_predictions.csv")
            if os.path.exists(model_file):
                df_pred = pd.read_csv(model_file)[target_columns]
                df_pred_list.append(df_pred)
            else:
                print(f"Warning: Prediction file not found: {model_file}")
        
        if not df_pred_list:
            print(f"Warning: No predictions found for fold {fold_idx}")
            continue
            
        # Average predictions across models for this fold
        df_pred_fold = pd.concat(df_pred_list).groupby(level=0).mean()
        
        # Calculate metrics for this fold
        fold_metrics = defaultdict(list)
        for column in target_columns:
            for metric, metric_func in metric_to_func.items():
                preds = df_pred_fold[column].tolist()
                targets = test_df[column].tolist()
                try:
                    metric_value = metric_func(targets, preds)
                    fold_metrics[metric].append(metric_value)
                except Exception as e:
                    print(f"Error calculating {metric} for fold {fold_idx}, column {column}: {e}")
                    fold_metrics[metric].append(np.nan)
        
        # Store fold results
        for metric in metrics:
            fold_results[metric].append(fold_metrics[metric])
    
    # Calculate summary statistics and format as mean ± std
    formatted_results = {}
    for metric in metrics:
        fold_metric_values = np.array(fold_results[metric])  # Shape: (num_folds, num_targets)
        
        means = np.nanmean(fold_metric_values, axis=0)
        stds = np.nanstd(fold_metric_values, axis=0, ddof=1)
        
        # Format as "mean ± std" strings
        formatted_values = []
        for mean_val, std_val in zip(means, stds):
            if np.isnan(mean_val) or np.isnan(std_val):
                formatted_values.append("NaN ± NaN")
            else:
                formatted_values.append(f"{mean_val:.{decimal_places}f} ± {std_val:.{decimal_places}f}")
        
        formatted_results[metric] = formatted_values
    
    # Create results DataFrame
    results_df = pd.DataFrame(formatted_results, index=target_columns)
    
    return {
        'summary': results_df,
        'fold_results': dict(fold_results)
    }

In [15]:
def evaluate_results_simple_with_nan(data_path, splits_path, result_dir, num_tasks, metrics, target_columns=None, num_folds=5, decimal_places=4):
    """
    Simple evaluation function for datasets with NaN values that computes mean ± std across folds.
    
    Args:
        data_path: Path to the dataset CSV file
        splits_path: Path to the multiple_splits.json file
        result_dir: Directory containing fold_X/model_Y/test_predictions.csv structure
        num_tasks: Number of target tasks
        metrics: List of metrics to compute
        target_columns: List of target column names (optional)
        num_folds: Number of folds (default: 5)
        decimal_places: Number of decimal places for formatting (default: 4)
    
    Returns:
        dict: Results with formatted mean ± std strings, including averaged metrics for multi-target datasets
    """
    df = pd.read_csv(data_path)
    with open(splits_path, "rb") as json_file:
        split_idxss = json.load(json_file)
    
    test_indices = parse_indices(split_idxss[0]["test"])
    test_df = df.iloc[test_indices]
    target_columns = test_df.keys()[-num_tasks:].tolist() if target_columns is None else target_columns
    
    metric_to_func = {metric: get_metric_func(metric) for metric in metrics}
    fold_results = defaultdict(list)
    
    for fold_idx in range(num_folds):
        fold_dir = os.path.join(result_dir, f"fold_{fold_idx}")
        
        if not os.path.exists(fold_dir):
            print(f"Warning: Fold directory {fold_dir} does not exist")
            continue
            
        # Find all model directories in this fold
        model_dirs = [d for d in os.listdir(fold_dir) 
                     if os.path.isdir(os.path.join(fold_dir, d)) and d.startswith('model_')]
        
        if not model_dirs:
            print(f"Warning: No model directories found in fold {fold_idx}")
            continue
        
        # Sort model directories by model index
        model_dirs.sort(key=lambda x: int(x.split('_')[1]))
        
        # Collect predictions from all models in this fold
        df_pred_list = []
        for model_dir in model_dirs:
            model_file = os.path.join(fold_dir, model_dir, "test_predictions.csv")
            if os.path.exists(model_file):
                df_pred = pd.read_csv(model_file)[target_columns]
                df_pred_list.append(df_pred)
            else:
                print(f"Warning: Prediction file not found: {model_file}")
        
        if not df_pred_list:
            print(f"Warning: No predictions found for fold {fold_idx}")
            continue
            
        df_pred_fold = pd.concat(df_pred_list).groupby(level=0).mean()
        
        fold_metrics = defaultdict(list)
        fold_averages = defaultdict(list)  # For averaging across targets
        
        for column in target_columns:
            for metric, metric_func in metric_to_func.items():
                preds = df_pred_fold[column].values
                targets = test_df[column].values
                
                # Vectorized NaN filtering
                valid_mask = ~np.isnan(targets)
                if np.any(valid_mask):
                    targets_clean = targets[valid_mask]
                    preds_clean = preds[valid_mask]
                    try:
                        metric_value = metric_func(targets_clean, preds_clean)
                        fold_metrics[metric].append(metric_value)
                        fold_averages[metric].append(metric_value)
                    except Exception as e:
                        print(f"Error calculating {metric} for fold {fold_idx}, column {column}: {e}")
                        fold_metrics[metric].append(np.nan)
                else:
                    fold_metrics[metric].append(np.nan)
        
        # Store fold results
        for metric in metrics:
            fold_results[metric].append(fold_metrics[metric])
    
    # Calculate per-target statistics and format as mean ± std
    formatted_results = {}
    for metric in metrics:
        fold_metric_values = np.array(fold_results[metric])
        means = np.nanmean(fold_metric_values, axis=0)
        stds = np.nanstd(fold_metric_values, axis=0, ddof=1)
        
        # Format as "mean ± std" strings
        formatted_values = []
        for mean_val, std_val in zip(means, stds):
            if np.isnan(mean_val) or np.isnan(std_val):
                formatted_values.append("NaN ± NaN")
            else:
                formatted_values.append(f"{mean_val:.{decimal_places}f} ± {std_val:.{decimal_places}f}")
        
        formatted_results[metric] = formatted_values
    
    # Calculate averaged statistics across all targets for multi-target datasets
    averaged_results = {}
    for metric in metrics:
        fold_metric_values = np.array(fold_results[metric])
        fold_averages = np.nanmean(fold_metric_values, axis=1)  # Average across targets for each fold
        
        avg_mean = np.nanmean(fold_averages)
        avg_std = np.nanstd(fold_averages, ddof=1)
        
        if np.isnan(avg_mean) or np.isnan(avg_std):
            averaged_results[metric] = "NaN ± NaN"
        else:
            averaged_results[metric] = f"{avg_mean:.{decimal_places}f} ± {avg_std:.{decimal_places}f}"
    
    # Create results DataFrame
    results_df = pd.DataFrame(formatted_results, index=target_columns)
    
    return {
        'summary': results_df,
        'averaged': averaged_results,  # Single row of averaged metrics
        'fold_results': dict(fold_results)
    }

In [16]:
def compare_methods_simple(rigr_results, native_results, dataset_name="Dataset"):
    """
    Create a simple side-by-side comparison of RIGR vs Native methods.
    
    Args:
        rigr_results: Results from evaluate_results_simple for RIGR
        native_results: Results from evaluate_results_simple for Native
        dataset_name: Name of the dataset for the markdown title
    
    Returns:
        str: Formatted markdown string with side-by-side comparison
    """
    
    # Get the summary DataFrames
    rigr_df = rigr_results['summary']
    native_df = native_results['summary']
    
    # Create comparison table
    markdown_lines = [f"# {dataset_name}\n"]
    
    # Check if we have the same metrics and targets
    metrics = rigr_df.columns.tolist()
    targets = rigr_df.index.tolist()
    
    if len(targets) == 1:
        # Single target - simple table
        markdown_lines.append("| Metric | RIGR | Native |")
        markdown_lines.append("|--------|------|--------|")
        
        for metric in metrics:
            rigr_val = rigr_df.loc[targets[0], metric]
            native_val = native_df.loc[targets[0], metric]
            markdown_lines.append(f"| {metric.upper()} | {rigr_val} | {native_val} |")
            
    else:
        # Multi-target - show per target
        for target in targets:
            markdown_lines.append(f"\n## {target}\n")
            markdown_lines.append("| Metric | RIGR | Native |")
            markdown_lines.append("|--------|------|--------|")
            
            for metric in metrics:
                rigr_val = rigr_df.loc[target, metric]
                native_val = native_df.loc[target, metric]
                markdown_lines.append(f"| {metric.upper()} | {rigr_val} | {native_val} |")
    
    return "\n".join(markdown_lines)


def compare_methods_simple_with_averaging(rigr_results, native_results, dataset_name="Dataset"):
    """
    Create a simple side-by-side comparison for datasets with many targets (like PCBA).
    Shows averaged metrics across all targets.
    
    Args:
        rigr_results: Results from evaluate_results_simple_with_nan for RIGR
        native_results: Results from evaluate_results_simple_with_nan for Native
        dataset_name: Name of the dataset for the markdown title
    
    Returns:
        str: Formatted markdown string with averaged comparison
    """
    
    # Get the averaged results
    rigr_avg = rigr_results['averaged']
    native_avg = native_results['averaged']
    
    # Create comparison table
    markdown_lines = [f"# {dataset_name}\n"]
    markdown_lines.append("*Averaged across all targets*\n")
    markdown_lines.append("| Metric | RIGR | Native |")
    markdown_lines.append("|--------|------|--------|")
    
    for metric in rigr_avg.keys():
        rigr_val = rigr_avg[metric]
        native_val = native_avg[metric]
        markdown_lines.append(f"| {metric.upper()} | {rigr_val} | {native_val} |")
    
    return "\n".join(markdown_lines)


def evaluate_and_compare_simple(data_path, splits_path, rigr_dir, native_dir, num_tasks, metrics, target_columns=None, decimal_places=4, dataset_name="Dataset"):
    """
    Convenience function to evaluate and compare RIGR vs Native methods with simple output.
    
    Args:
        data_path: Path to dataset CSV
        splits_path: Path to multiple_splits.json
        rigr_dir: Directory containing RIGR results
        native_dir: Directory containing Native/Chemprop results
        num_tasks: Number of target tasks
        metrics: List of metrics to compute
        target_columns: Target column names (optional)
        decimal_places: Number of decimal places for formatting
        dataset_name: Name of the dataset
    
    Returns:
        str: Formatted markdown comparison
    """

    rigr_results = evaluate_results_simple(
        data_path, splits_path, rigr_dir, num_tasks, metrics, target_columns, decimal_places=decimal_places
    )
    
    native_results = evaluate_results_simple(
        data_path, splits_path, native_dir, num_tasks, metrics, target_columns, decimal_places=decimal_places
    )
    
    comparison_md = compare_methods_simple(rigr_results, native_results, dataset_name)
    
    return {
        'rigr': rigr_results,
        'native': native_results,
        'markdown': comparison_md
    }


def evaluate_and_compare_simple_with_nan(data_path, splits_path, rigr_dir, native_dir, num_tasks, metrics, target_columns=None, decimal_places=4, dataset_name="Dataset"):
    """
    Convenience function for datasets with NaN values (like PCBA).
    
    Args:
        data_path: Path to dataset CSV
        splits_path: Path to multiple_splits.json
        rigr_dir: Directory containing RIGR results
        native_dir: Directory containing Native/Chemprop results
        num_tasks: Number of target tasks
        metrics: List of metrics to compute
        target_columns: Target column names (optional)
        decimal_places: Number of decimal places for formatting
        dataset_name: Name of the dataset
    
    Returns:
        dict: Contains results and formatted markdown
    """

    rigr_results = evaluate_results_simple_with_nan(
        data_path, splits_path, rigr_dir, num_tasks, metrics, target_columns, decimal_places=decimal_places
    )
    
    native_results = evaluate_results_simple_with_nan(
        data_path, splits_path, native_dir, num_tasks, metrics, target_columns, decimal_places=decimal_places
    )
    
    comparison_md = compare_methods_simple_with_averaging(rigr_results, native_results, dataset_name)
    
    return {
        'rigr': rigr_results,
        'native': native_results,
        'markdown': comparison_md
    }

## Barrier Cycloadd

In [33]:
# Dataset configuration
dataset_name = "Barrier Cycloadd"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/barrier_cycloadd"
data_path = "/home/akshatz/bond_order_free/barriers_cycloadd/dataset/cycloadd_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["G_act"]
decimal_places = 3  # Customize the number of decimal places

# Evaluate and compare methods
try:
    results = evaluate_and_compare_simple(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/barrier_cycloadd/results.md

Preview of results:
# Barrier Cycloadd

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 2.807 ± 0.032 | 2.651 ± 0.042 |
| RMSE | 4.172 ± 0.031 | 3.955 ± 0.040 |
| R2 | 0.837 ± 0.002 | 0.854 ± 0.003 |


## Barrier E2

In [34]:
# Dataset configuration
dataset_name = "Barrier E2"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/barrier_e2"
data_path = "/home/akshatz/bond_order_free/barriers_e2/dataset/e2_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["ea"]
decimal_places = 3  # Customize the number of decimal places

# Evaluate and compare methods
try:
    results = evaluate_and_compare_simple(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/barrier_e2/results.md

Preview of results:
# Barrier E2

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 2.679 ± 0.054 | 2.501 ± 0.027 |
| RMSE | 3.855 ± 0.034 | 3.698 ± 0.053 |
| R2 | 0.885 ± 0.002 | 0.894 ± 0.003 |


## Barrier RDB7

In [35]:
# Dataset configuration
dataset_name = "Barrier RDB7"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/barrier_rdb7"
data_path = "/home/akshatz/bond_order_free/barriers_rdb7/dataset/rdb7_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["ea"]
decimal_places = 3  # Customize the number of decimal places

# Evaluate and compare methods
try:
    results = evaluate_and_compare_simple(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/barrier_rdb7/results.md

Preview of results:
# Barrier RDB7

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 4.341 ± 0.020 | 4.097 ± 0.021 |
| RMSE | 7.895 ± 0.093 | 7.336 ± 0.022 |
| R2 | 0.932 ± 0.002 | 0.941 ± 0.000 |


## Barrier RGD1-CNHO

In [ ]:
# Dataset configuration
dataset_name = "Barrier RGD1-CNHO"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/barrier_rgd1_cnho"
data_path = "/home/akshatz/bond_order_free/barriers_rgd1/dataset/rgd1_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["ea"]
decimal_places = 3  # Customize the number of decimal places

# Evaluate and compare methods
try:
    results = evaluate_and_compare_simple(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/barrier_rgd1_cnho/results.md

Preview of results:
# Barrier RGD1-CNHO

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 6.212 ± 0.060 | 5.762 ± 0.010 |
| RMSE | 10.413 ± 0.070 | 9.601 ± 0.010 |
| R2 | 0.877 ± 0.002 | 0.896 ± 0.000 |


## Barrier SN2

In [36]:
# Dataset configuration
dataset_name = "Barrier SN2"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/barrier_sn2"
data_path = "/home/akshatz/bond_order_free/barriers_sn2/dataset/sn2_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["ea"]
decimal_places = 3  # Customize the number of decimal places

# Evaluate and compare methods
try:
    results = evaluate_and_compare_simple(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/barrier_sn2/results.md

Preview of results:
# Barrier SN2

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 2.549 ± 0.031 | 2.635 ± 0.017 |
| RMSE | 3.485 ± 0.035 | 3.497 ± 0.012 |
| R2 | 0.921 ± 0.002 | 0.920 ± 0.001 |


## HIV

In [37]:
# Dataset configuration
dataset_name = "HIV"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/hiv"
data_path = "/home/akshatz/bond_order_free/hiv/dataset/hiv_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["prc-auc", "roc-auc", "ap"]
target_columns = ["HIV_active"]
decimal_places = 3  # Customize the number of decimal places

# Evaluate and compare methods
try:
    results = evaluate_and_compare_simple(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/hiv/results.md

Preview of results:
# HIV

| Metric | RIGR | Native |
|--------|------|--------|
| PRC-AUC | 0.293 ± 0.021 | 0.282 ± 0.014 |
| ROC-AUC | 0.782 ± 0.013 | 0.773 ± 0.014 |
| AP | 0.296 ± 0.019 | 0.286 ± 0.013 |


## PCQM4MV2

In [38]:
# Dataset configuration
dataset_name = "PCQM4MV2"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/pcqm4mv2"
data_path = "/home/akshatz/bond_order_free/pcqm4mv2/dataset/pcqm4mv2_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["homolumogap"]
decimal_places = 4  # Customize the number of decimal places

# Evaluate and compare methods
try:
    results = evaluate_and_compare_simple(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/pcqm4mv2/results.md

Preview of results:
# PCQM4MV2

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 0.0969 ± 0.0003 | 0.0919 ± 0.0002 |
| RMSE | 0.1635 ± 0.0003 | 0.1524 ± 0.0004 |
| R2 | 0.9803 ± 0.0001 | 0.9829 ± 0.0001 |


## QM9 Gap

In [39]:
# Dataset configuration
dataset_name = "QM9 Gap"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/qm9/qm9_gap"
data_path = "/home/akshatz/bond_order_free/qm9/dataset/qm9_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["gap"]
decimal_places = 5  # Customize the number of decimal places

# Evaluate and compare methods
try:
    results = evaluate_and_compare_simple(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/qm9/qm9_gap/results.md

Preview of results:
# QM9 Gap

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 0.00314 ± 0.00001 | 0.00312 ± 0.00002 |
| RMSE | 0.00602 ± 0.00013 | 0.00592 ± 0.00016 |
| R2 | 0.98376 ± 0.00069 | 0.98429 ± 0.00084 |


## QM9 U0

In [41]:
# Dataset configuration
dataset_name = "QM9 U0"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/qm9/qm9_u0"
data_path = "/home/akshatz/bond_order_free/qm9/dataset/qm9_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["u0_atom"]
decimal_places = 3  # Customize the number of decimal places

# Evaluate and compare methods
try:
    results = evaluate_and_compare_simple(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/qm9/qm9_u0/results.md

Preview of results:
# QM9 U0

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 1.081 ± 0.017 | 1.027 ± 0.021 |
| RMSE | 2.508 ± 0.080 | 2.448 ± 0.058 |
| R2 | 1.000 ± 0.000 | 1.000 ± 0.000 |


## QM9 Multitask

In [42]:
# Dataset configuration
dataset_name = "QM9 Multitask"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/qm9/qm9_multitask"
data_path = "/home/akshatz/bond_order_free/qm9/dataset/qm9_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 12
metrics = ["mae", "rmse", "r2"]
target_columns = None  # Auto-detect all 12 QM9 targets
decimal_places = 6  # Customize the number of decimal places

# Evaluate and compare methods
try:
    results = evaluate_and_compare_simple(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/qm9/qm9_multitask/results.md

Preview of results:
# QM9 Multitask


## mu

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 0.331338 ± 0.000823 | 0.340634 ± 0.001452 |
| RMSE | 0.589317 ± 0.006579 | 0.603310 ± 0.006580 |
| R2 | 0.851852 ± 0.003310 | 0.844734 ± 0.003390 |

## alpha

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 0.220731 ± 0.001032 | 0.227535 ± 0.002627 |
| RMSE | 0.547334 ± 0.017582 | 0.597339 ± 0.038735 |
| R2 | 0.995326 ± 0.000298 | 0.994418 ± 0.000711 |

## homo

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 0.002417 ± 0.000012 | 0.002458 ± 0.000008 |
| RMSE | 0.004256 ± 0.000079 | 0.004309 ± 0.000060 |
| R2 | 0.962385 ± 0.001394 | 0.961445 ± 0.001068 |

## lumo

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 0.002376 ± 0.000013 | 0.002375 ± 0.000007 |
| RMSE | 0.004094 ± 0.000032 | 0.004109 ± 0.000041 |
| R2 | 0.992341 ± 0.000119 | 0.99

## UV/Vis

In [43]:
# Dataset configuration
dataset_name = "UV/Vis"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/uv_vis"
data_path = "/home/akshatz/bond_order_free/multi_molecule/dataset/mult_mol_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 1
metrics = ["mae", "rmse", "r2"]
target_columns = ["peakwavs_max"]
decimal_places = 3  # Customize the number of decimal places

# Evaluate and compare methods
try:
    results = evaluate_and_compare_simple(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/uv_vis/results.md

Preview of results:
# UV/Vis

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 16.000 ± 0.050 | 16.573 ± 0.216 |
| RMSE | 31.276 ± 0.576 | 31.259 ± 0.604 |
| R2 | 0.911 ± 0.003 | 0.912 ± 0.003 |


## PCBA Random

In [17]:
# Function for handling datasets with NaN targets (like PCBA)
def evaluate_results_multi_fold_with_nan(data_path, splits_path, result_dir, num_tasks, metrics, target_columns=None, num_folds=5):
    """
    Version of evaluate_results_multi_fold that handles NaN values in targets.
    Automatically detects the number of models in each fold.
    """
    df = pd.read_csv(data_path)
    with open(splits_path, "rb") as json_file:
        split_idxss = json.load(json_file)
    
    test_indices = parse_indices(split_idxss[0]["test"])
    test_df = df.iloc[test_indices]
    target_columns = test_df.keys()[-num_tasks:].tolist() if target_columns is None else target_columns
    
    metric_to_func = {metric: get_metric_func(metric) for metric in metrics}
    fold_results = defaultdict(list)
    
    for fold_idx in range(num_folds):
        fold_dir = os.path.join(result_dir, f"fold_{fold_idx}")
        
        # Automatically detect the number of models in this fold
        if not os.path.exists(fold_dir):
            print(f"Warning: Fold directory {fold_dir} does not exist")
            continue
            
        # Find all model directories in this fold
        model_dirs = [d for d in os.listdir(fold_dir) 
                     if os.path.isdir(os.path.join(fold_dir, d)) and d.startswith('model_')]
        
        if not model_dirs:
            print(f"Warning: No model directories found in fold {fold_idx}")
            continue
        
        # Sort model directories by model index
        model_dirs.sort(key=lambda x: int(x.split('_')[1]))
        
        # Collect predictions from all models in this fold
        df_pred_list = []
        for model_dir in model_dirs:
            model_file = os.path.join(fold_dir, model_dir, "test_predictions.csv")
            if os.path.exists(model_file):
                df_pred = pd.read_csv(model_file)[target_columns]
                df_pred_list.append(df_pred)
            else:
                print(f"Warning: Prediction file not found: {model_file}")
        
        if not df_pred_list:
            print(f"Warning: No predictions found for fold {fold_idx}")
            continue
            
        df_pred_fold = pd.concat(df_pred_list).groupby(level=0).mean()
        
        fold_metrics = defaultdict(list)
        for column in target_columns:
            for metric, metric_func in metric_to_func.items():
                preds = df_pred_fold[column].values
                targets = test_df[column].values
                
                # Vectorized NaN filtering (much faster)
                valid_mask = ~np.isnan(targets)
                if np.any(valid_mask):
                    targets_clean = targets[valid_mask]
                    preds_clean = preds[valid_mask]
                    try:
                        metric_value = metric_func(targets_clean, preds_clean)
                        fold_metrics[metric].append(metric_value)
                    except Exception as e:
                        print(f"Error calculating {metric} for fold {fold_idx}, column {column}: {e}")
                        fold_metrics[metric].append(np.nan)
                else:
                    fold_metrics[metric].append(np.nan)
        
        for metric in metrics:
            fold_results[metric].append(fold_metrics[metric])
    
    # Calculate summary statistics (same as before)
    summary_results = {}
    for metric in metrics:
        fold_metric_values = np.array(fold_results[metric])
        summary_results[f"{metric}_mean"] = np.nanmean(fold_metric_values, axis=0)
        summary_results[f"{metric}_std"] = np.nanstd(fold_metric_values, axis=0, ddof=1)
        summary_results[f"{metric}_folds"] = fold_metric_values
    
    summary_df = pd.DataFrame({
        metric + "_mean": summary_results[f"{metric}_mean"] for metric in metrics
    }, index=target_columns)
    
    for metric in metrics:
        summary_df[f"{metric}_std"] = summary_results[f"{metric}_std"]
    
    return {
        'summary': summary_df,
        'fold_results': dict(fold_results),
        'raw_results': summary_results
    }


def compare_methods_statistical_averaged(method1_results, method2_results, metrics, method1_name="Method1", method2_name="Method2", alpha=0.05):
    """
    Compare two methods using statistical tests on averaged metrics across all targets.
    This is appropriate for multi-target datasets like PCBA where we want one comparison per metric.
    
    Args:
        method1_results: Results from evaluate_results_multi_fold_with_nan for method 1
        method2_results: Results from evaluate_results_multi_fold_with_nan for method 2
        metrics: List of metrics to compare
        method1_name: Name of method 1 for display
        method2_name: Name of method 2 for display
    
    Returns:
        pd.DataFrame: Comparison results with p-values (one row per metric)
    """
    comparison_results = []
    
    for metric in metrics:
        
        method1_folds = method1_results['raw_results'][f"{metric}_folds"]  # Shape: (num_folds, num_targets)
        method2_folds = method2_results['raw_results'][f"{metric}_folds"]  # Shape: (num_folds, num_targets)
        
        # Average across all targets for each fold
        method1_fold_averages = np.nanmean(method1_folds, axis=1)  # Shape: (num_folds,)
        method2_fold_averages = np.nanmean(method2_folds, axis=1)  # Shape: (num_folds,)
        
        # Check for NaN values
        if np.any(np.isnan(method1_fold_averages)) or np.any(np.isnan(method2_fold_averages)):
            print(f"  Warning: NaN values found for averaged {metric}")
            t_stat, p_value = np.nan, np.nan
        elif len(method1_fold_averages) < 2 or len(method2_fold_averages) < 2:
            print(f"  Warning: Not enough samples for averaged {metric}")
            t_stat, p_value = np.nan, np.nan
        elif np.allclose(method1_fold_averages, method2_fold_averages):
            print(f"  Warning: Identical values for averaged {metric}")
            t_stat, p_value = 0.0, 1.0
        else:
            try:
                # Perform paired t-test on averaged values
                t_stat, p_value = ttest_rel(method1_fold_averages, method2_fold_averages)
            except Exception as e:
                print(f"  Error in t-test for averaged {metric}: {e}")
                t_stat, p_value = np.nan, np.nan
        
        # Calculate means and effect size
        method1_mean = np.nanmean(method1_fold_averages)
        method2_mean = np.nanmean(method2_fold_averages)
        method1_std = np.nanstd(method1_fold_averages, ddof=1)
        method2_std = np.nanstd(method2_fold_averages, ddof=1)
        
        # Cohen's d for effect size
        pooled_std = np.sqrt((method1_std**2 + method2_std**2) / 2)
        cohens_d = (method1_mean - method2_mean) / pooled_std if pooled_std > 0 and not np.isnan(pooled_std) else 0
        
        comparison_results.append({
            'metric': metric,
            f'{method1_name}_mean': method1_mean,
            f'{method1_name}_std': method1_std,
            f'{method2_name}_mean': method2_mean,
            f'{method2_name}_std': method2_std,
            't_statistic': t_stat,
            'p_value': p_value,
            'cohens_d': cohens_d,
            'significant': p_value < alpha if not np.isnan(p_value) else False
        })
    
    return pd.DataFrame(comparison_results)


def evaluate_and_compare_methods_with_nan(data_path, splits_path, rigr_dir, native_dir, num_tasks, metrics, target_columns=None):
    """
    Convenience function to evaluate and compare RIGR vs Native methods for datasets with NaN targets.
    Uses averaged statistical comparison for multi-target datasets.
    
    Args:
        data_path: Path to dataset CSV
        splits_path: Path to multiple_splits.json
        rigr_dir: Directory containing RIGR results
        native_dir: Directory containing Native/Chemprop results
        num_tasks: Number of target tasks
        metrics: List of metrics to compute
        target_columns: Target column names (optional)
    
    Returns:
        dict: Contains individual results and comparison
    """

    rigr_results = evaluate_results_multi_fold_with_nan(
        data_path, splits_path, rigr_dir, num_tasks, metrics, target_columns
    )
    
    native_results = evaluate_results_multi_fold_with_nan(
        data_path, splits_path, native_dir, num_tasks, metrics, target_columns
    )
    
    # Use averaged comparison for multi-target datasets
    comparison = compare_methods_statistical_averaged(
        rigr_results, native_results, metrics, "RIGR", "Native"
    )
    
    return {
        'rigr': rigr_results,
        'native': native_results,
        'comparison': comparison
    }

In [18]:
# Dataset configuration
dataset_name = "PCBA Random"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/pcba/pcba_random"
data_path = "/home/akshatz/bond_order_free/pcba_random/dataset/pcba_random_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 128  # PCBA has 128 tasks
metrics = ["prc-auc", "roc-auc", "ap"]
target_columns = None  # Will auto-detect all 128 PCBA targets
decimal_places = 3  # Use 3 decimal places for PCBA

# Evaluate and compare methods (using NaN-aware function for PCBA)
try:
    results = evaluate_and_compare_simple_with_nan(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
        
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/pcba/pcba_random/results.md

Preview of results:
# PCBA Random

*Averaged across all targets*

| Metric | RIGR | Native |
|--------|------|--------|
| PRC-AUC | 0.198 ± 0.005 | 0.211 ± 0.002 |
| ROC-AUC | 0.900 ± 0.002 | 0.905 ± 0.001 |
| AP | 0.204 ± 0.005 | 0.216 ± 0.002 |


## PCBA Random NaN

In [19]:
# Dataset configuration
dataset_name = "PCBA Random NaN"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/pcba/pcba_random_nan"
data_path = "/home/akshatz/bond_order_free/pcba_random_nan/dataset/pcba_random_nan_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 128  # PCBA has 128 tasks
metrics = ["prc-auc", "roc-auc", "ap"]
target_columns = None  # Will auto-detect all 128 PCBA targets
decimal_places = 3  # Use 3 decimal places for PCBA

# Evaluate and compare methods (using NaN-aware function for PCBA)
try:
    results = evaluate_and_compare_simple_with_nan(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
        
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/pcba/pcba_random_nan/results.md

Preview of results:
# PCBA Random NaN

*Averaged across all targets*

| Metric | RIGR | Native |
|--------|------|--------|
| PRC-AUC | 0.363 ± 0.010 | 0.373 ± 0.009 |
| ROC-AUC | 0.899 ± 0.001 | 0.904 ± 0.002 |
| AP | 0.369 ± 0.010 | 0.378 ± 0.008 |


## PCBA Scaffold

In [20]:
# Dataset configuration
dataset_name = "PCBA Scaffold"
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/pcba/pcba_scaffold"
data_path = "/home/akshatz/bond_order_free/pcba_scaffold/dataset/pcba_scaffold_data.csv"
splits_path = f"{base_dir}/multiple_splits.json"
rigr_dir = f"{base_dir}/rigr/results"
native_dir = f"{base_dir}/native/results"

results_md_path = os.path.join(base_dir, "results.md")

num_tasks = 127  # PCBA Scaffold has 127 tasks
metrics = ["prc-auc", "roc-auc", "ap"]
target_columns = None  # Will auto-detect all PCBA targets
decimal_places = 3  # Use 3 decimal places for PCBA

# Evaluate and compare methods (using NaN-aware function for PCBA)
try:
    results = evaluate_and_compare_simple_with_nan(
        data_path=data_path,
        splits_path=splits_path,
        rigr_dir=rigr_dir,
        native_dir=native_dir,
        num_tasks=num_tasks,
        metrics=metrics,
        target_columns=target_columns,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/pcba/pcba_scaffold/results.md

Preview of results:
# PCBA Scaffold

*Averaged across all targets*

| Metric | RIGR | Native |
|--------|------|--------|
| PRC-AUC | 0.299 ± 0.010 | 0.302 ± 0.012 |
| ROC-AUC | 0.877 ± 0.005 | 0.887 ± 0.004 |
| AP | 0.303 ± 0.010 | 0.306 ± 0.012 |


# SAMPL

In [ ]:
# For the SAMPL datasets.. I do not want to look at the test_predictions.csv.. Instead I have individual predictions for each of the 25 predictions in a single file (like /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/SAMPL/native/results_sampl_production/pred_SAMPL7_individual.csv)
# Those are probably just first 5 for fold 0, next 5 for fold 1 and so on.. I want to do same analysis as we did for above (simply get the error bar on each metric -- average predictions within each fold and average the metric calculated for each fold)


In [46]:
def evaluate_sampl_simple(test_no, rigr_result_dir, native_result_dir, metrics, num_folds=5, num_models_per_fold=5, decimal_places=4):
    """
    Evaluate SAMPL datasets with the individual prediction file structure.
    
    Args:
        test_no: SAMPL test number (6, 7, or 9)
        rigr_result_dir: Directory containing RIGR results
        native_result_dir: Directory containing Native results  
        metrics: List of metrics to compute
        num_folds: Number of folds (default: 5)
        num_models_per_fold: Number of models per fold (default: 5)
        decimal_places: Number of decimal places for formatting
    
    Returns:
        dict: Results with formatted mean ± std strings for both methods
    """
    
    def evaluate_single_method(result_dir, method_name):
        # Find the individual predictions file
        files = glob.glob(os.path.join(result_dir, '**', f"pred_SAMPL{test_no}_individual.csv"), recursive=True)
        if len(files) != 1:
            raise ValueError(f"Expected 1 file for {method_name}, found {len(files)}: {files}")
        
        df = pd.read_csv(files[0])
        
        # Get targets - different column name for SAMPL9
        if test_no == 9:
            targets = df["new_logPexp_reviewed"].tolist()
        else:
            targets = df["logP mean"].tolist()
        
        # Get predictions for each model (pred_0, pred_1, ..., pred_24)
        pred_columns = [f"pred_0_model_{i}" for i in range(num_folds * num_models_per_fold)]
        
        # Reshape predictions into folds
        # First 5 predictions are fold 0, next 5 are fold 1, etc.
        fold_results = []
        metric_to_func = {metric: get_metric_func(metric) for metric in metrics}
        
        for fold_idx in range(num_folds):
            # Get prediction columns for this fold
            fold_pred_cols = pred_columns[fold_idx * num_models_per_fold:(fold_idx + 1) * num_models_per_fold]
            
            # Average predictions across models in this fold
            fold_preds = df[fold_pred_cols].mean(axis=1).tolist()
            
            # Calculate metrics for this fold
            fold_metrics = {}
            for metric, metric_func in metric_to_func.items():
                try:
                    metric_value = metric_func(targets, fold_preds)
                    fold_metrics[metric] = metric_value
                except Exception as e:
                    print(f"Error calculating {metric} for {method_name} fold {fold_idx}: {e}")
                    fold_metrics[metric] = np.nan
            
            fold_results.append(fold_metrics)
        
        # Calculate mean ± std across folds
        formatted_results = {}
        for metric in metrics:
            fold_values = [fold_result[metric] for fold_result in fold_results]
            mean_val = np.nanmean(fold_values)
            std_val = np.nanstd(fold_values, ddof=1)
            
            if np.isnan(mean_val) or np.isnan(std_val):
                formatted_results[metric] = "NaN ± NaN"
            else:
                formatted_results[metric] = f"{mean_val:.{decimal_places}f} ± {std_val:.{decimal_places}f}"
        
        return formatted_results
    
    # Evaluate both methods
    rigr_results = evaluate_single_method(rigr_result_dir, "RIGR")
    native_results = evaluate_single_method(native_result_dir, "Native")
    
    return {
        'rigr': rigr_results,
        'native': native_results
    }


def compare_sampl_simple(rigr_results, native_results, test_no, dataset_name=None):
    """
    Create a simple side-by-side comparison for SAMPL datasets.
    
    Args:
        rigr_results: Results from evaluate_sampl_simple for RIGR
        native_results: Results from evaluate_sampl_simple for Native
        test_no: SAMPL test number
        dataset_name: Optional dataset name override
    
    Returns:
        str: Formatted markdown string with side-by-side comparison
    """
    if dataset_name is None:
        dataset_name = f"SAMPL{test_no}"
    
    # Create comparison table
    markdown_lines = [f"# {dataset_name}\n"]
    markdown_lines.append("| Metric | RIGR | Native |")
    markdown_lines.append("|--------|------|--------|")
    
    # Get all metrics (should be the same for both)
    metrics = list(rigr_results.keys())
    
    for metric in metrics:
        rigr_val = rigr_results[metric]
        native_val = native_results[metric]
        markdown_lines.append(f"| {metric.upper()} | {rigr_val} | {native_val} |")
    
    return "\n".join(markdown_lines)


def evaluate_and_compare_sampl(test_no, rigr_result_dir, native_result_dir, metrics, decimal_places=4, dataset_name=None):
    """
    Convenience function to evaluate and compare SAMPL datasets.
    
    Args:
        test_no: SAMPL test number (6, 7, or 9)
        rigr_result_dir: Directory containing RIGR results
        native_result_dir: Directory containing Native results
        metrics: List of metrics to compute
        decimal_places: Number of decimal places for formatting
        dataset_name: Optional dataset name override
    
    Returns:
        dict: Contains results and formatted markdown
    """
    
    # Evaluate both methods
    results = evaluate_sampl_simple(
        test_no=test_no,
        rigr_result_dir=rigr_result_dir,
        native_result_dir=native_result_dir,
        metrics=metrics,
        decimal_places=decimal_places
    )
    
    # Create comparison markdown
    comparison_md = compare_sampl_simple(
        rigr_results=results['rigr'],
        native_results=results['native'],
        test_no=test_no,
        dataset_name=dataset_name
    )
    
    return {
        'rigr': results['rigr'],
        'native': results['native'],
        'markdown': comparison_md
    }

In [47]:
# Dataset configuration for SAMPL6
dataset_name = "SAMPL6"
test_no = 6
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/SAMPL"
rigr_result_dir = f"{base_dir}/rigr/results_sampl_production"
native_result_dir = f"{base_dir}/native/results_sampl_production"

results_md_path = os.path.join(base_dir, f"sampl{test_no}_results.md")

metrics = ["mae", "rmse", "r2"]
decimal_places = 3

# Evaluate and compare methods
try:
    results = evaluate_and_compare_sampl(
        test_no=test_no,
        rigr_result_dir=rigr_result_dir,
        native_result_dir=native_result_dir,
        metrics=metrics,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/SAMPL/sampl6_results.md

Preview of results:
# SAMPL6

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 0.301 ± 0.038 | 0.301 ± 0.023 |
| RMSE | 0.359 ± 0.043 | 0.382 ± 0.024 |
| R2 | 0.706 ± 0.070 | 0.671 ± 0.041 |


In [48]:
# Dataset configuration for SAMPL7
dataset_name = "SAMPL7"
test_no = 7
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/SAMPL"
rigr_result_dir = f"{base_dir}/rigr/results_sampl_production"
native_result_dir = f"{base_dir}/native/results_sampl_production"

results_md_path = os.path.join(base_dir, f"sampl{test_no}_results.md")

metrics = ["mae", "rmse", "r2"]
decimal_places = 3

# Evaluate and compare methods
try:
    results = evaluate_and_compare_sampl(
        test_no=test_no,
        rigr_result_dir=rigr_result_dir,
        native_result_dir=native_result_dir,
        metrics=metrics,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/SAMPL/sampl7_results.md

Preview of results:
# SAMPL7

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 0.321 ± 0.023 | 0.396 ± 0.024 |
| RMSE | 0.451 ± 0.022 | 0.588 ± 0.040 |
| R2 | 0.537 ± 0.045 | 0.214 ± 0.106 |


In [49]:
# Dataset configuration for SAMPL9
dataset_name = "SAMPL9"
test_no = 9
base_dir = "/home/akshatz/bond_order_free/rigr/chemprop_benchmarks/SAMPL"
rigr_result_dir = f"{base_dir}/rigr/results_sampl_production"
native_result_dir = f"{base_dir}/native/results_sampl_production"

results_md_path = os.path.join(base_dir, f"sampl{test_no}_results.md")

metrics = ["mae", "rmse", "r2"]
decimal_places = 3

# Evaluate and compare methods
try:
    results = evaluate_and_compare_sampl(
        test_no=test_no,
        rigr_result_dir=rigr_result_dir,
        native_result_dir=native_result_dir,
        metrics=metrics,
        decimal_places=decimal_places,
        dataset_name=dataset_name
    )
    
    # Write the simple markdown comparison to file
    with open(results_md_path, "w") as f:
        f.write(results['markdown'])
    
    print(f"Results saved to {results_md_path}")
    print("\nPreview of results:")
    print(results['markdown'])
    
except Exception as e:
    print(f"Error during evaluation: {e}")
    import traceback
    traceback.print_exc()

Results saved to /home/akshatz/bond_order_free/rigr/chemprop_benchmarks/SAMPL/sampl9_results.md

Preview of results:
# SAMPL9

| Metric | RIGR | Native |
|--------|------|--------|
| MAE | 0.956 ± 0.041 | 0.927 ± 0.016 |
| RMSE | 1.087 ± 0.045 | 1.103 ± 0.019 |
| R2 | 0.764 ± 0.020 | 0.758 ± 0.008 |
